**What is the RAG system?**

Retrieval-Augmented Generation (RAG) is an AI architecture that combines the power of:

- Retrieval → Finding relevant information from external knowledge sources.
  
- Generation → Using a Large Language Model (LLM) to generate accurate, natural-language answers based on the retrieved information.

### **Why we create a RAG System?**

Retrieval systems (RAG) give LLM systems access to factual, access-control, timely information.

**1. RAG Reduces Hallucination:**

- Hallucination, or the generation of incorrect or misleading information, is a common challenge in large language models.
- The RAG framework significantly reduces this issue by combining the power of retrieval and generation.
- when an LLM doesn't know the correct answer but still generates one that sounds confident.

`Example:` In the financial services industry, providing accurate information on investment options is crucial because it directly impacts customers' purchasing decisions and financial well-being. RAG can help ensure that the information generated about stocks, bonds, or mutual funds is well-grounded in factual data, reducing the risk of misleading clients and potentially harmful financial decisions.

**2. Cost Effective Alternative:**

`Example:` Banks often need to assess the creditworthiness of potential borrowers. Fine-tuning pre-trained language models to analyse credit histories can be resource-intensive. RAG architecture offers a cost-effective alternative by retrieving relevant financial data and credit history information from existing databases, combining this with pre-trained language models to generate accurate credit assessments. This approach saves both time and money while still providing reliable results for banks.

**3. Credible and Accurate Response:**

`Example:` In customer support, providing accurate and helpful responses is essential for maintaining customer trust, as it demonstrates the company's commitment to providing reliable information and support. The RAG technique is able to do this very effectively by retrieving data from catalogues, policies, and past customer interactions to generate context-aware insights, ensuring that customers receive reliable information on product features, returns, and other inquiries.

**4. Domain Specific Information:**

`Example:` In the legal industry, clients often require advice specific to their case or jurisdiction because different legal systems have unique rules and regulations, and understanding these nuances is crucial for effective legal representation. RAG can access domain-specific knowledge bases, such as local statutes and case law, to provide tailored information relevant to clients' legal needs.

https://www.advancinganalytics.co.uk/blog/2023/11/7/10-reasons-why-you-need-to-implement-rag-a-game-changer-in-ai#3._credible_and_accurate_responses

**RAG Practival Usecase**

1. Document Question Answering Systems
2. Conversational agents
3. Real-time Event Commentary
4. Content Generation
5. Personalised Recommendation
6. Virtual Assistants

**Imports**

In [ ]:
from google.colab import userdata

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

**API Key**

In [ ]:
EURI_API_KEY = userdata.get("EURI_API_KEY")

**Load Document**

In [ ]:
loader = TextLoader(
    "state_of_the_union.txt",
    encoding="utf8"
)

In [ ]:
documents = loader.load()

In [ ]:
print(documents[0].page_content[:500])





  

<!DOCTYPE html>
<html
  lang="en"
  
  data-color-mode="auto" data-light-theme="light" data-dark-theme="dark"
  data-a11y-animated-images="system" data-a11y-link-underlines="true"
  
  >




  <head>
    <meta charset="utf-8">
  <link rel="dns-prefetch" href="https://github.githubassets.com">
  <link rel="dns-prefetch" href="https://avatars.githubusercontent.com">
  <link rel="dns-prefetch" href="https://github-cloud.s3.amazonaws.com">
  <link rel="dns-prefetch" href="https://user-images


**Split Document**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [ ]:
chunks = text_splitter.split_documents(documents)

In [ ]:
print(chunks[3].page_content)

media="all" rel="stylesheet" href="https://github.githubassets.com/assets/dark_high_contrast-6739a26cef6aaf8e.css" /><link data-color-theme="light" crossorigin="anonymous" media="all" rel="stylesheet" data-href="https://github.githubassets.com/assets/light-62b06818b06b09b7.css" /><link data-color-theme="light_high_contrast" crossorigin="anonymous" media="all" rel="stylesheet" data-href="https://github.githubassets.com/assets/light_high_contrast-44cd405df9340c5c.css" /><link


**Embeddings**

In [ ]:
embeddings = OpenAIEmbeddings(
    api_key=EURI_API_KEY,
    base_url="https://api.euron.one/api/v1/euri",
    model="text-embedding-3-small"
)

In [ ]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [ ]:
retriever = vectorstore.as_retriever()

**Prompt**

In [ ]:
template = """
You are an assistant for question answering tasks.

Use the retrieved context below to answer the user's question.

If you don't know the answer, simply say "I don't know."

Keep the answer concise (maximum 10 sentences).

Question:
{question}

Context:
{context}

Answer:
"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

**LLM**

In [ ]:
llm = ChatOpenAI(
    model="gemini-2.5-pro",
    api_key=EURI_API_KEY,
    base_url="https://api.euron.one/api/v1/euri",
    temperature=0
)

**Output Parser**

In [ ]:
output_parser = StrOutputParser()

**RAG Chain**

In [ ]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | output_parser
)


**Ask Question**

In [ ]:
response = rag_chain.invoke(
    "How is the United States supporting Ukraine economically and militarily?"
)

In [ ]:
print(response)

In [ ]:
import requests
from google.colab import userdata

EURI_API_KEY = userdata.get("EURI_API_KEY")

url = "https://api.euron.one/api/v1/euri/chat/completions"

headers = {
    "Authorization": f"Bearer {EURI_API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": "gemini-2.5-pro",
    "messages": [
        {
            "role": "user",
            "content": "Hello"
        }
    ]
}

response = requests.post(url, headers=headers, json=payload)

print(response.status_code)
print(response.text)

200
{"id":"r5Jpar6aKJCKjuMP7KOemQo","object":"chat.completion","created":1785303732,"model":"gemini-2.5-pro","choices":[{"index":0,"message":{"role":"assistant","content":"Hello there! How can I help you today?"},"finish_reason":"stop"}],"usage":{"prompt_tokens":2,"completion_tokens":10,"total_tokens":488}}
